<img src="logo.png" alt="Vegeta" width="240">

# Myropods: Persephone — a segmented crawler for chimneys and tight dangerous spaces

**Myropods** are a family of segmented walking robots: a chain of identical hollow segments joined by
two-axis joints, **four legs on every segment** (two pairs, like a myriapod), so each segment can stand
and brace on its own and the controller treats it as one simple unit, a head with lights and a camera, a
tether at the tail. The members differ in the number and size of the segments: **Persephone** (this
notebook) goes down into the dark — twelve 60 × 44 × 34 mm segments, 48 legs, one metre long, built for
flues and ducts of 120–210 mm with elbows and vertical runs; **Cleopatra** (notebook 18) is the agile
three-segment walker of the same design file, scaled up.

Three versions of Persephone, in order:

| version | what it adds | sections |
|---|---|---|
| **v1** | the crawler as in the sketch: segments, legs, head, tether | 1–7: fit, elbows, bracing and climbing, strength, heat, comms and power, gait dynamics, fatigue |
| **v2** | transport into the chimney: the crawler coiled around its tether drum in a carry case, the flue-mouth insertion guide, the insertion sequence (push in, anchor and pull, emergency retreat) | 8 |
| **v3** | two pincers on the head for nests and debris | 9 |

```
CAD (Dedalus): segment, leg, head, chain (straight, bent, coiled) ─► masses, envelope, elbow kinematics
bracing: friction on soot ─► normal force per leg ─► servo torque ─► servo class ─► mass (a fixed-point loop)
flue height ─► tether weight ─► climb power ─► tether voltage drop
operating cases ─► FEA of the leg and the segment shell (Talos), pins and tether by hand
warm flue ─► lumped thermal model ─► minutes inside
links (tether power, fibre, LR mesh, relay, local logs) ─► budgets
leg modes ─► frequency diagram vs stepping harmonics
missions ─► bracing histories ─► rainflow (Chronos) ─► leg fatigue ─► life
v2: case + guide CAD, insertion forces, case FEA     v3: pincers CAD, grip, head moment, pincer FEA
```
Every input is explicit and coarse (friction on soot, servo data, an assumed S-N curve). Compare, record,
then put a real segment in a real flue.

In [ ]:
import json, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Arc
from tqdm.auto import tqdm
from vegeta import dedalus, talos, chronos, mellonia
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.mellonia import viz as mviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM

RUNS = Path("_runs/persephone"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
design_file = RUNS / "myropod.py"
shutil.copy(Path("designs/myropod.py"), design_file)
myropod = dedalus.load_design(f"{design_file}:Myropod")
pd.DataFrame(myropod.params.table()).set_index("name")

## 1. Version 1 — the crawler as in the sketch

The default parameters *are* Persephone. One segment, one leg and the head are generated on their own for
FEA and printing; the whole chain straight, through an elbow, and coiled (what version 2 needs).

In [ ]:
p = myropod.resolve()
seg, leg, head = myropod.generate(part="segment"), myropod.generate(part="leg"), myropod.generate(part="head")
crawler = myropod.generate()
bent = myropod.generate(bend_pitch_deg=-18.0, bend_from=4)                 # nose up: climbing out of an elbow
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.05) for k, g in (("segment", seg), ("leg", leg), ("head", head), ("crawler", crawler))}
PITCH = myropod.pitch_length(p); N = p["n_segments"]; REACH = myropod.leg_reach(p)
LENGTH = p["head_length"] + N * PITCH
print(f"{N} segments, {4 * N} legs | pitch {PITCH:.0f} mm | length {LENGTH:.0f} mm | stance width {REACH['stance_width']:.0f} mm | belly clearance {REACH['drop_z']:.0f} mm")
dviz.show(dviz.plot3d(crawler))

In [ ]:
dviz.show(dviz.plot3d(bent))

In [ ]:
fig = dviz.plot_sections(seg, normal="x", positions=[0.0, myropod.hip_x(p)], cols=2)        # through the shell, through the hip bosses
fig = dviz.plot_sections(leg, normal="x", positions=[0.0], cols=1)

In [ ]:
RHO_PA12CF = 1.1e-3                      # g/mm^3 printed PA12-CF (shell, legs)
SERVOS = pd.DataFrame([                  # candidate leg servos: metal gear, 7.4 V (inputs from datasheets)
    ("sub-micro 9 g", 9.0, 0.22, 0.9), ("micro 13 g", 13.0, 0.32, 1.1), ("micro-metal 20 g", 20.0, 0.55, 1.4), ("mini 32 g", 32.0, 1.10, 2.0), ("standard 55 g", 55.0, 2.20, 3.0),
], columns=["servo", "mass_g", "stall_Nm", "stall_A"]).set_index("servo")
JOINT_SERVO = {"name": "mini 32 g (pitch + yaw per joint)", "mass_g": 32.0, "stall_Nm": 1.10}
def mass_budget(servo):
    s = SERVOS.loc[servo]
    per_segment = pd.Series({
        "shell + hatch + rib (from CAD)": seg.volume * RHO_PA12CF,
        "legs 4x (from CAD)": 4 * leg.volume * RHO_PA12CF,
        f"leg servos 8x ({servo})": 8 * s["mass_g"],
        "joint servos 2x (pitch, yaw)": 2 * JOINT_SERVO["mass_g"],
        "segment PCB, IMU, contact sensors": 14.0,
        "wiring, pins, pads": 12.0,
    })
    head_g = head.volume * RHO_PA12CF + 2 * 6.0 + 25.0 + 8.0 + 30.0      # lights, camera, IMU, radio + antenna
    tail_g = 25.0                                                          # tether gland, strain relief
    return per_segment, N * per_segment.sum() + head_g + tail_g
per_segment, m_total_g = mass_budget("micro-metal 20 g")
print(f"first pass with 20 g servos: {per_segment.sum():.0f} g per segment, {m_total_g / 1000:.2f} kg crawler")
per_segment.round(1)

### Fit: which flues, which elbows

Bracing means pressing feet on both walls at once. The legs reach from folded (`hip_angle` −30°, knee 120°)
to stretched (hip 0°, knee 0°); the body plus two reaches gives the bracing range of flue diameters. Larger
flues are walked on the floor of an inclined run, or need a longer-legged member of the family.

An elbow is a polygon problem: a chain with pitch `L` turns a bend of centre-line radius `R` with a joint
angle `φ = L / R`; the joints allow ±45°. A sharp (mitre) 90° corner passes a rigid segment of length `l`
and height `h` in a duct wider than `(l + h) / √2`. The drawing shows the chain in a 90° elbow of a 150 mm
flue.

In [ ]:
JOINT_RANGE_DEG = 45.0                                                    # per joint, pitch and yaw (input)
def reach_y(hip_deg, knee_deg):
    return p["femur_length"] * math.cos(math.radians(hip_deg)) + p["tibia_length"] * math.cos(math.radians(knee_deg))
D_MIN_BRACE = p["seg_width"] + 2 * 6 + 2 * max(reach_y(-30, 120), p["femur_length"] * math.cos(math.radians(-30))) + p["foot_diameter"]   # legs folded: knees out
D_MAX_BRACE = p["seg_width"] + 2 * 6 + 2 * reach_y(0, 0) * 0.95 + p["foot_diameter"]                                                        # 95 % of the full stretch
R_MIN_BEND = PITCH / (2 * math.sin(math.radians(JOINT_RANGE_DEG) / 2))
D_MIN_MITRE = (p["seg_length"] + p["seg_height"]) / math.sqrt(2)
flues = {}
for D in (100, 120, 150, 180, 200, 250, 300):
    R_bend = D                                                             # a tight elbow: centre-line radius = one diameter (input)
    phi = math.degrees(PITCH / R_bend)
    flues[f"{D} mm"] = {"bracing possible": D_MIN_BRACE <= D <= D_MAX_BRACE, "legs for bracing: hip/knee [deg]": "folded" if D < D_MIN_BRACE else ("stretched" if D > D_MAX_BRACE else "ok"),
                        "joint angle in a tight 90° elbow [deg]": phi, "elbow ok": phi <= JOINT_RANGE_DEG, "mitre corner ok": D >= D_MIN_MITRE,
                        "sagging body in a horizontal run: legs reach the floor": REACH["drop_z"] >= 0}
print(f"bracing range {D_MIN_BRACE:.0f}–{D_MAX_BRACE:.0f} mm | tightest bend radius {R_MIN_BEND:.0f} mm | mitre corner needs ≥ {D_MIN_MITRE:.0f} mm")
pd.DataFrame(flues).T

In [ ]:
D_FLUE = 150.0
R_b = D_FLUE
fig, ax = plt.subplots(figsize=(6.5, 6.5))
for r in (R_b - D_FLUE / 2, R_b + D_FLUE / 2):                              # the elbow walls
    ax.add_patch(Arc((0, 0), 2 * r, 2 * r, theta1=0, theta2=90, color="#5a6a7a", lw=2))
ax.plot([R_b - D_FLUE / 2, R_b + D_FLUE / 2], [-300, -300], color="#5a6a7a"); ax.plot([R_b - D_FLUE / 2] * 2, [-300, 0], color="#5a6a7a", lw=2); ax.plot([R_b + D_FLUE / 2] * 2, [-300, 0], color="#5a6a7a", lw=2)
ax.plot([-300, 0], [R_b - D_FLUE / 2] * 2, color="#5a6a7a", lw=2); ax.plot([-300, 0], [R_b + D_FLUE / 2] * 2, color="#5a6a7a", lw=2)
# the chain: straight up the vertical run, around the bend with phi per joint, then horizontal
phi = PITCH / R_b
pts, ang = [(R_b, -250.0)], math.pi / 2
pos = np.array(pts[0])
for i in range(N + 1):
    if pos[1] >= 0 and ang > 1e-6:
        ang = max(ang - phi, 0.0)
    d = np.array([math.cos(ang), math.sin(ang)])
    pos = pos + d * PITCH * (-1)
    pts.append(tuple(pos))
    ang_use = ang
pts = np.array(pts)
ax.plot(pts[:, 0], pts[:, 1], "o-", color="#1565c0", lw=3, ms=5, label=f"chain, pitch {PITCH:.0f} mm, {math.degrees(phi):.0f}° per joint")
ax.set(aspect="equal", xlim=(-320, 320), ylim=(-320, 320), title=f"Persephone in a 90° elbow of a {D_FLUE:.0f} mm flue (centre-line radius {R_b:.0f} mm)")
ax.legend(loc="lower left"); ax.grid(alpha=0.2)

## 2. Bracing and climbing a vertical flue

In a vertical run everything hangs on friction: the braced feet press the walls with normal forces `N`,
and `μ ΣN` must carry the crawler plus the tether hanging below it (`TETHER_G_PER_M` × height), with a
safety factor. Soot on brick is slippery (`MU_SOOT`). A wave gait keeps two thirds of the legs in stance.
Each stance leg's hip servo holds `N × reach`, its knee servo `N × tibia lever`. The servo class decides the
mass, and the mass decides `N`: a fixed-point loop over the servo table picks the lightest servo that holds
with the torque safety factor.

In [ ]:
G = 9.81
MU_SOOT, MU_CLEAN = 0.30, 0.60             # rubber pads on sooty / clean brick (inputs)
SF_SLIP, SF_TORQUE = 2.0, 1.5               # friction and servo-torque safety factors (inputs)
TETHER_G_PER_M = 45.0                       # 8 mm tether: 2 power cores, fibre, Kevlar braid (input)
HEIGHT_M = 15.0                             # the vertical run (input)
STANCE_FRACTION = 2 / 3                     # wave gait (input)
D_WORK = 150.0                              # the flue the servos are sized for (input)
def legs_for(D):
    """Hip and knee angles that put the foot on the wall of a flue of diameter D (both legs of a pair reach the same wall distance)."""
    wall = (D - p["seg_width"]) / 2 - 6 - p["foot_diameter"] / 2        # hip to wall
    best = min(((abs(reach_y(h, k) - wall), h, k) for h in np.arange(-30, 60, 2.0) for k in np.arange(20, 120, 2.0)), key=lambda t: t[0])
    return best[1], best[2]
def bracing(m_kg, D=D_WORK, mu=MU_SOOT, height_m=HEIGHT_M):
    W = (m_kg + TETHER_G_PER_M / 1000 * height_m) * G
    n_stance = STANCE_FRACTION * 4 * N
    N_leg = SF_SLIP * W / (mu * n_stance)
    hip, knee = legs_for(D)
    lever_hip = reach_y(hip, knee)                                        # normal force x horizontal reach (the pad pushes straight at the wall)
    lever_knee = p["tibia_length"] * math.cos(math.radians(knee))
    friction_leg = W / n_stance                                           # the share of the weight each stance leg carries along the wall
    tau_hip = (N_leg * lever_hip + friction_leg * (p["femur_length"] * math.sin(math.radians(hip)) + p["tibia_length"] * math.sin(math.radians(knee)))) / 1000
    tau_knee = (N_leg * lever_knee + friction_leg * p["tibia_length"] * math.sin(math.radians(knee))) / 1000
    return {"W_total_N": W, "N_leg_N": N_leg, "friction_leg_N": friction_leg, "hip_deg": hip, "knee_deg": knee, "tau_hip_Nm": tau_hip, "tau_knee_Nm": tau_knee}
choice = None
for servo in SERVOS.index:                                                # lightest first
    per_segment, m_total_g = mass_budget(servo)
    b = bracing(m_total_g / 1000)
    if SF_TORQUE * max(b["tau_hip_Nm"], b["tau_knee_Nm"]) <= SERVOS.loc[servo, "stall_Nm"]:
        choice = servo; break
per_segment, M_G = mass_budget(choice); M_CRAWLER = M_G / 1000
B = bracing(M_CRAWLER)
print(f"servo: {choice} ({SERVOS.loc[choice, 'stall_Nm']} Nm) -> {per_segment.sum():.0f} g per segment, crawler {M_CRAWLER:.2f} kg | at {HEIGHT_M:.0f} m: hanging {B['W_total_N']:.0f} N, {B['N_leg_N']:.1f} N per braced foot, hip {B['tau_hip_Nm']:.2f} Nm, knee {B['tau_knee_Nm']:.2f} Nm")
rows = {}
for D in (120, 150, 180, 200):
    for h in (0.0, 7.5, 15.0):
        rows[f"{D} mm flue, {h:.0f} m up"] = bracing(M_CRAWLER, D=D, height_m=h)
brace_tab = pd.DataFrame(rows).T
brace_tab["servo SF"] = SERVOS.loc[choice, "stall_Nm"] / brace_tab[["tau_hip_Nm", "tau_knee_Nm"]].max(axis=1)
brace_tab.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
hs = np.linspace(0, 25, 26)
for D, col in ((120, "#2e7d32"), (150, "#1565c0"), (200, "#c62828")):
    tau = [max(bracing(M_CRAWLER, D=D, height_m=h)["tau_hip_Nm"], bracing(M_CRAWLER, D=D, height_m=h)["tau_knee_Nm"]) for h in hs]
    ax.plot(hs, tau, color=col, label=f"{D} mm flue, soot μ={MU_SOOT}")
    tau_c = [max(bracing(M_CRAWLER, D=D, mu=MU_CLEAN, height_m=h)["tau_hip_Nm"], bracing(M_CRAWLER, D=D, mu=MU_CLEAN, height_m=h)["tau_knee_Nm"]) for h in hs]
    ax.plot(hs, tau_c, "--", color=col, label=f"{D} mm flue, clean μ={MU_CLEAN}")
ax.axhline(SERVOS.loc[choice, "stall_Nm"] / SF_TORQUE, color="k", ls="-.", label=f"{choice}: stall / SF {SF_TORQUE}")
ax.set(xlabel="height climbed [m] (tether hanging below)", ylabel="worst leg servo torque [N m]", title="holding torque per braced leg vs height and flue"); ax.legend(fontsize=7); ax.grid(alpha=0.3)

### Speed, power, the tether

A wave gait with a `STEP_MM` step at `STEP_HZ` moves the body `step × f × (1 − stance fraction)` — slow,
as a chimney crawler should be. The electrical power is the servos holding the walls (a plain servo at a
fraction `f` of its stall torque draws about `f × I_stall` — for 64 braced servos that is hundreds of watts
and a thermal disaster, so the leg gears are self-locking and holding costs `HOLD_CURRENT` of that), plus
the swing legs and the electronics. The tether
carries it at `V_TETHER` over its length: copper cross-section against voltage drop and loss.

In [ ]:
STEP_MM, STEP_HZ = 25.0, 1.0                                               # inputs
V_TETHER, V_SERVO = 48.0, 7.4
CU_MM2, RHO_CU = 0.75, 0.0178                                              # tether power cores [mm^2], ohm mm^2 / m
P_ELECTRONICS = 12.0                                                       # head computer, camera, lights, segment PCBs [W]
HOLD_CURRENT = 0.15                                                        # self-locking leg gears (worm / lead screw): holding a braced leg costs this fraction of the torque-proportional current (input)
v_climb = STEP_MM / 1000 * STEP_HZ * (1 - STANCE_FRACTION)
def power(height_m, D=D_WORK, mu=MU_SOOT):
    b = bracing(M_CRAWLER, D=D, mu=mu, height_m=height_m)
    s = SERVOS.loc[choice]
    f_hold = max(b["tau_hip_Nm"], b["tau_knee_Nm"]) / s["stall_Nm"]
    n_stance = STANCE_FRACTION * 4 * N
    p_hold = n_stance * 2 * f_hold * s["stall_A"] * V_SERVO * HOLD_CURRENT  # two servos per stance leg, self-locking gears
    p_swing = (4 * N - n_stance) * 2 * 0.25 * s["stall_A"] * V_SERVO         # swing legs at a quarter of stall current
    p_lift = b["W_total_N"] * v_climb / 0.3                                 # lifting the weight at 30 % drive efficiency
    p_total = (p_hold + p_swing + p_lift) / 0.9 + P_ELECTRONICS              # DC-DC at 90 %
    return {"hold_W": p_hold, "swing_W": p_swing, "lift_W": p_lift, "total_W": p_total}
rows = {}
for L_tether in (30.0, 50.0):
    for h in (0.0, 15.0):
        pw = power(h)
        R = 2 * RHO_CU * L_tether / CU_MM2
        I = pw["total_W"] / V_TETHER
        rows[f"{L_tether:.0f} m tether, {h:.0f} m up"] = {**pw, "current_A": I, "voltage_drop_V": I * R, "tether_loss_W": I**2 * R, "drop_%": I * R / V_TETHER * 100}
climb_time_min = HEIGHT_M / v_climb / 60
print(f"climb speed {v_climb * 1000:.0f} mm/s -> {HEIGHT_M:.0f} m in {climb_time_min:.0f} min | energy for the climb ≈ {power(HEIGHT_M / 2)['total_W'] * climb_time_min / 60:.0f} Wh")
pd.DataFrame(rows).T.round(2)

## 3. Strength (Talos): leg, segment shell, pins, tether

The leg in its own frame (hip bore held, forces on the foot pad): bracing on the working flue, bracing in
the widest flue with the leg stretched, a slip-catch (one segment's four legs catch the whole hanging
weight when the rest let go), a stuck leg at servo stall, and a step on the floor. The segment shell (front neck held): the squeeze of four braced legs, the
chain hanging from one joint while the lower part is out of an elbow, and a dead pull-out on the tether.
Pins in double shear and the tether against its breaking strength by hand.

In [ ]:
PA12CF = talos.Material("PA12-CF (printed)", youngs_modulus=3500.0, poissons_ratio=0.40, density=1.1e-9, yield_strength=60.0,
                        source="nominal datasheet, flat orientation; assume 30 % less across layers")
STEEL_PIN = talos.Material("steel pin", youngs_modulus=210000.0, poissons_ratio=0.30, density=7.85e-9, yield_strength=400.0, source="handbook")
fy, dz, fd = REACH["foot_y"], REACH["drop_z"], p["foot_diameter"]
LEG_REGIONS = [talos.SurfacesInBox("hip", (-6, -3.0, -3.0, 6, 3.0, 3.0)),
               talos.SurfacesInBox("foot", (-fd, fy - fd, -dz - fd, fd, fy + fd, -dz + fd))]
def leg_model(loads, name, masses=()):
    return talos.StructuralModel(cad["leg"].artifacts["step"], "mm-N-MPa", PA12CF, LEG_REGIONS, [talos.FixedSupport("hip")], loads,
                                 talos.MeshSettings(element_size=1.6, min_element_size=0.6, curvature_points=10), name=name, masses=list(masses))
s = SERVOS.loc[choice]
F_STALL_HIP = s["stall_Nm"] * 1000 / REACH["foot_y"]                       # the force a stuck hip servo can put on the wall
b_wide = bracing(M_CRAWLER, D=D_MAX_BRACE - 2, height_m=HEIGHT_M)
LEG_CASES = {   # forces on the foot pad in the leg frame: -fy = the wall pushes the foot back towards the body, fx = along the flue (the body's x)
    "bracing, 150 mm flue, 15 m up": dict(fy=-B["N_leg_N"], fx=B["friction_leg_N"]),
    "bracing, widest flue (leg stretched)": dict(fy=-b_wide["N_leg_N"], fx=b_wide["friction_leg_N"]),
    "slip-catch: one segment's four legs catch the hanging weight": dict(fy=-B["N_leg_N"], fx=B["W_total_N"] / 4),
    "stuck leg at servo stall": dict(fy=-F_STALL_HIP, fx=0.3 * F_STALL_HIP),
    "walking on the floor, one foot of four": dict(fz=M_CRAWLER * G / N / 4 * 2.0),   # a 2 g step, the segment on its own four legs
}
leg_model([talos.Force("foot", fz=1.0)], "mesh").mesh(RUNS / "leg_mesh", progress=True)
leg_results = {}
for name, f in tqdm(LEG_CASES.items()):
    key = "leg_" + name.replace(" ", "_").replace(",", "").replace("(", "").replace(")", "")
    mdl = leg_model([talos.Force("foot", **f)], key); mdl.mesh(RUNS / key); leg_results[name] = mdl.solve(RUNS / key)
print({k: r.status for k, r in leg_results.items()})

In [ ]:
Ls, W, H, zc = p["seg_length"], p["seg_width"], p["seg_height"], p["seg_height"] / 2
g_ = p["joint_gap"]
SEG_REGIONS = [talos.SurfacesInBox("front_neck", (Ls / 2 + 0.5, -8, zc - 8, Ls / 2 + g_ / 2 + 3, 8, zc + 8)),
               talos.SurfacesInBox("rear_neck", (-Ls / 2 - g_ / 2 - 3, -9, zc - 9, -Ls / 2 - 0.5, 9, zc + 9)),
               talos.SurfacesInBox("boss_left", (-Ls / 2, W / 2 + 0.5, zc - 9, Ls / 2, W / 2 + 7, zc + 9)),
               talos.SurfacesInBox("boss_right", (-Ls / 2, -W / 2 - 7, zc - 9, Ls / 2, -W / 2 - 0.5, zc + 9))]
def seg_model(loads, name):
    return talos.StructuralModel(cad["segment"].artifacts["step"], "mm-N-MPa", PA12CF, SEG_REGIONS, [talos.FixedSupport("front_neck")], loads,
                                 talos.MeshSettings(element_size=2.5, min_element_size=1.0), name=name)
W_SEG = per_segment.sum() / 1000 * G
F_HANG = (N - 1) * W_SEG + TETHER_G_PER_M / 1000 * HEIGHT_M * G            # everything behind the first segment, plus the tether
MU_JAM = 0.8
F_PULLOUT = M_CRAWLER * G + MU_JAM * 4 * N * F_STALL_HIP * 0.5             # dead pull-out: weight + friction of half the legs jammed at stall
SEG_CASES = {
    "braced: four legs squeeze the shell": [talos.Force("boss_left", fy=-2 * B["N_leg_N"] * 1.5), talos.Force("boss_right", fy=2 * B["N_leg_N"] * 1.5)],
    "hanging chain from one joint": [talos.Force("rear_neck", fz=-F_HANG)],
    "dead pull-out on the tether": [talos.Force("rear_neck", fx=-F_PULLOUT)],
}
seg_model([talos.Force("rear_neck", fz=1.0)], "mesh").mesh(RUNS / "seg_mesh", progress=True)
seg_results = {}
for name, loads in tqdm(SEG_CASES.items()):
    key = "seg_" + name.split(":")[0].replace(" ", "_")
    mdl = seg_model(loads, key); mdl.mesh(RUNS / key); seg_results[name] = mdl.solve(RUNS / key)
print({k: r.status for k, r in seg_results.items()}, f"| hanging {F_HANG:.0f} N, pull-out {F_PULLOUT:.0f} N")

In [ ]:
def pin_stress(F, d): return math.sqrt(3) * F / (2 * math.pi * d**2 / 4)        # double shear, von Mises
TETHER_BREAK_N = 2500.0                                                           # Kevlar braid, 8 mm tether (input, supplier)
hand = {
    "joint pin — hanging chain": {"load": f"{F_HANG:.0f} N shear", "max_von_mises_MPa": pin_stress(F_HANG, p["joint_pin_diameter"]), "SF_yield": STEEL_PIN.yield_strength / pin_stress(F_HANG, p["joint_pin_diameter"]), "deflection_mm": float("nan")},
    "joint pin — dead pull-out": {"load": f"{F_PULLOUT:.0f} N shear", "max_von_mises_MPa": pin_stress(F_PULLOUT, p["joint_pin_diameter"]), "SF_yield": STEEL_PIN.yield_strength / pin_stress(F_PULLOUT, p["joint_pin_diameter"]), "deflection_mm": float("nan")},
    "leg pin — slip-catch, one of four legs": {"load": f"{B['W_total_N'] / 4:.0f} N shear", "max_von_mises_MPa": pin_stress(B["W_total_N"] / 4, p["joint_pin_diameter"]), "SF_yield": STEEL_PIN.yield_strength / pin_stress(B["W_total_N"] / 4, p["joint_pin_diameter"]), "deflection_mm": float("nan")},
    "tether — dead pull-out": {"load": f"{F_PULLOUT:.0f} N tension", "max_von_mises_MPa": float("nan"), "SF_yield": TETHER_BREAK_N / F_PULLOUT, "deflection_mm": float("nan")},
}
def rows(results, label, cases):
    return pd.DataFrame({f"{label} — {k}": {"load": ", ".join(f"{kk} {vv:.0f} N" for kk, vv in cases[k].items()) if isinstance(cases[k], dict) else "see the case",
                                            "max_von_mises_MPa": r.metrics.get("max_von_mises"), "SF_yield": r.metrics.get("safety_factor_yield"), "deflection_mm": r.metrics.get("max_displacement")} for k, r in results.items()}).T
static = pd.concat([rows(leg_results, "leg", LEG_CASES), rows(seg_results, "segment", SEG_CASES), pd.DataFrame(hand).T])
fig, ax = plt.subplots(figsize=(9, 5.5))
static["SF_yield"].astype(float).plot.barh(ax=ax, color=["#1565c0"] * len(LEG_CASES) + ["#2e7d32"] * len(SEG_CASES) + ["#6d4c41"] * len(hand))
ax.axvline(1.5, color="#c62828", ls="--", label="SF 1.5 (printed parts)"); ax.set(xlabel="safety factor on yield (tether: on breaking strength)", xscale="log"); ax.legend(); ax.grid(alpha=0.3, axis="x")
static.round(2)

In [ ]:
worst_leg = static.loc[[k for k in static.index if k.startswith("leg —")], "SF_yield"].astype(float).idxmin().split(" — ")[1]
tviz.show(tviz.plot_results(leg_results[worst_leg], field="von_mises"))

In [ ]:
tviz.show(tviz.plot_results(seg_results["hanging chain from one joint"], field="von_mises"))

## 4. A warm flue: how long inside

Chimneys are inspected cold, but a flue that was in use yesterday is still warm and a wrong one is hot.
A lumped model per segment: the PCB and its battery backup as one thermal mass behind the shell and an
insulating liner, heated by the wall through radiation and convection on the shell and by its own
dissipation, cooled by nothing. The servos sit in the hip bosses, outside the liner: their losses go to
the shell and the flue air, not to the electronics. Time until the electronics reach their limit, per wall temperature.

In [ ]:
T_LIMIT, T_START = 70.0, 20.0                       # electronics limit and start temperature [°C]
M_ELEC, C_ELEC = 0.04, 900                           # PCB + backup cell per segment [kg], J/(kg K)
A_SHELL = seg.surface_area * 1e-6 * 0.7              # outer shell area [m^2] (70 % faces the wall and air)
H_OUT = 12.0                                         # W/(m^2 K): radiation + convection from the wall and flue air to the shell (input)
LINER_MM, K_LINER = 4.0, 0.025                       # aerogel blanket liner inside the shell (inputs)
U = 1 / (1 / (H_OUT * A_SHELL) + LINER_MM / 1000 / (K_LINER * A_SHELL) + 1 / (5.0 * A_SHELL))    # wall -> shell -> liner -> inner air -> electronics [W/K]
P_SELF = 0.3                                                                                    # the segment PCB's own dissipation behind the liner [W]
t = np.linspace(0, 120 * 60, 2000)
tau = M_ELEC * C_ELEC / U
fig, ax = plt.subplots(figsize=(8, 4.5))
thermal_tab = {}
for Tw in (40, 60, 80, 120, 160, 200):
    T_inf = Tw + P_SELF / U
    T = T_inf + (T_START - T_inf) * np.exp(-t / tau)
    ax.plot(t / 60, T, label=f"wall {Tw} °C")
    over = np.where(T >= T_LIMIT)[0]
    thermal_tab[f"wall {Tw} °C"] = {"steady electronics [°C]": T_inf, "minutes to the limit": (t[over[0]] / 60 if len(over) else float("inf"))}
ax.axhline(T_LIMIT, color="#c62828", ls="--", label=f"limit {T_LIMIT:.0f} °C"); ax.axvline(climb_time_min, color="#888", ls=":", label=f"one {HEIGHT_M:.0f} m climb")
ax.set(xlabel="time inside [min]", ylabel="electronics temperature [°C]", title=f"lumped thermal model per segment (τ = {tau / 60:.1f} min, U = {U:.3f} W/K, self-heating {P_SELF:.2f} W)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
pd.DataFrame(thermal_tab).T.round(1)

## 5. Communication and power: the panel in the sketch

Five links, each with a job and a fallback. The tether carries power and a fibre; the long-range mesh
(LoRa, 868 MHz) is the telemetry fallback when the fibre is cut or the crawler is let off the tether; a relay
node dropped at the flue mouth puts a second hop between the brick and the operator; local logs keep the
video when nothing else works. The link budget treats the flue as a lossy brick duct.

In [ ]:
LINKS = pd.DataFrame([
    ("tether power", "48 V DC, 2 x 0.75 mm²", f"{power(HEIGHT_M)['total_W']:.0f} W at 15 m up", "battery in the head: 10 min retreat", "cut tether"),
    ("fibre data", "single-mode fibre in the tether", "1 Gb/s, 4K video + telemetry", "LR mesh (telemetry), local logs (video)", "cut or kinked fibre"),
    ("mesh LR", "LoRa 868 MHz, SF12", "0.3–5 kb/s telemetry, commands", "relay hop, store and forward", "brick attenuation in deep flues"),
    ("relay", "dropped node at the flue mouth", "LoRa + Wi-Fi to the operator", "second relay at the hearth", "node battery"),
    ("local logs", "64 GB in the head", f"{64e9 * 8 / 20e6 / 3600:.0f} h of 20 Mb/s video", "—", "recovered with the crawler"),
], columns=["link", "what", "capacity", "fallback", "fails when"]).set_index("link")
TX_DBM, RX_SENS_DBM = 14.0, -137.0
BRICK_WALL_DB, DUCT_DB_PER_M = 10.0, 2.5                                   # inputs: one brick wall, and the brick duct as a lossy waveguide
def fspl(d_m, f_hz=868e6): return 20 * math.log10(d_m) + 20 * math.log10(f_hz) - 147.55
budget = {}
for depth in (5.0, 10.0, 15.0):
    for relay in (False, True):
        d_free = 15.0 if relay else 60.0                                   # relay at the flue mouth vs the operator in the street
        walls = 1 if relay else 3
        loss = fspl(d_free) + DUCT_DB_PER_M * depth + BRICK_WALL_DB * walls
        budget[f"{depth:.0f} m deep, {'with' if relay else 'no'} relay"] = {"path loss [dB]": loss, "margin [dB]": TX_DBM - loss - RX_SENS_DBM, "link": "ok" if TX_DBM - loss - RX_SENS_DBM > 10 else "marginal" if TX_DBM - loss - RX_SENS_DBM > 0 else "lost"}
display(LINKS)
pd.DataFrame(budget).T.round(1)

## 6. Gait dynamics: leg modes against the stepping harmonics

A leg is a small cantilever with a servo at its root; a foot slapping the wall at the step frequency and its
harmonics must not find the leg's bending mode. The frequency diagram runs over the step frequency.

In [ ]:
leg_modal = leg_model([], "leg_modal", masses=[talos.PointMass("foot", 0.002e-3)])
leg_modal.mesh(RUNS / "leg_modal"); modes = leg_modal.solve_modes(RUNS / "leg_modal", n_modes=3)
freqs = modes.metrics["frequencies_hz"]
structure = chronos.Structure(tuple(freqs), damping_ratio=0.04)
fs = np.linspace(0.3, 3.0, 28)
fig, ax = plt.subplots(figsize=(8, 4.5))
for k in (1, 2, 4, 8, 16):
    ax.plot(fs, k * fs, label=f"step x{k}")
for i, f in enumerate(freqs):
    ax.axhline(f, color="#c62828", ls="--", lw=0.8); ax.text(fs[0], f, f" leg mode {i + 1}: {f:.0f} Hz", color="#c62828", va="bottom", fontsize=8)
ax.axhline(50, color="#1565c0", ls="-.", lw=0.8); ax.text(fs[-1], 50, "servo PWM 50 Hz ", color="#1565c0", ha="right", va="bottom", fontsize=8)
ax.axvline(STEP_HZ, color="#888", ls=":"); ax.text(STEP_HZ, 0.4, " climbing step rate", rotation=90, va="bottom", fontsize=8)
ax.set(xlabel="step frequency [Hz]", ylabel="frequency [Hz]", yscale="log", ylim=(0.3, 5000), title="frequency diagram: stepping harmonics vs the leg modes"); ax.legend(fontsize=7, loc="upper left"); ax.grid(alpha=0.3, which="both")
print(f"leg modes {[round(f, 1) for f in freqs]} Hz | margin of the 16th step harmonic to the nearest mode {structure.margin(16 * STEP_HZ):.2f}")
tviz.show(tviz.plot_mode(modes, mode=1))

## 7. Cyclic loads: bracing cycles over a mission (Chronos), fatigue of the leg

Every step a leg lets go, swings, lands and presses again: one bracing cycle of amplitude `N` (pattern
`brace_y`) with its friction share (`brace_x`), plus the odd slip-catch. Histories are synthesised from the
gait, counted with rainflow and scaled to the mission; unit FEA cases turn them into damage per node.

In [ ]:
MISSIONS = {   # name: (flue D, height m, minutes, mu, slip-catches per mission)
    "cold 150 mm flue, 15 m": (150.0, 15.0, 2 * climb_time_min + 10, MU_SOOT, 2),
    "tight 120 mm flue with elbows": (120.0, 8.0, 2 * 8 / v_climb / 60 + 15, MU_SOOT, 6),
    "clean 180 mm duct, horizontal": (180.0, 0.0, 45.0, MU_CLEAN, 0),
}
SIM_S = 90.0
def brace_history(D, h, minutes, mu, catches, seed):
    rng = np.random.default_rng(seed); dt = 2e-3; tt = np.arange(0, SIM_S, dt)
    b = bracing(M_CRAWLER, D=D, mu=mu, height_m=h / 2)
    Ny, Fx = np.zeros_like(tt), np.zeros_like(tt)
    T = 1 / STEP_HZ
    for k in range(int(SIM_S * STEP_HZ)):
        on = (tt >= k * T) & (tt < k * T + STANCE_FRACTION * T)
        ph = (tt[on] - k * T) / (STANCE_FRACTION * T)
        amp = b["N_leg_N"] * rng.normal(1.0, 0.2)
        Ny[on] = amp * np.clip(np.sin(np.pi * ph) * 1.3, 0, 1); Fx[on] = b["friction_leg_N"] * rng.normal(1.0, 0.2)
    for k in rng.choice(int(SIM_S * STEP_HZ), size=max(1, round(catches / minutes * SIM_S / 60)), replace=False):
        on = (tt >= k * T) & (tt < k * T + 0.08)
        Fx[on] = b["W_total_N"] / 4 * np.sin(np.pi * (tt[on] - k * T) / 0.08)       # a slip-catch shared by the segment's four legs
    return tt, Ny, Fx
hist = {name: brace_history(*spec, seed=i) for i, (name, spec) in enumerate(MISSIONS.items())}
fig, axes = plt.subplots(len(hist), 1, figsize=(13, 7), sharex=True)
for ax, (name, (tt, Ny, Fx)) in zip(axes, hist.items()):
    ax.plot(tt, Ny, lw=0.6, label="brace_y (normal force)"); ax.plot(tt, Fx, lw=0.6, label="brace_x (along the flue)"); ax.set(title=name, ylabel="N", xlim=(0, 12)); ax.legend(fontsize=8); ax.grid(alpha=0.3)
axes[-1].set(xlabel="time [s] (first 12 s of the simulated history)"); fig.tight_layout()

In [ ]:
spectra = {}
for name, (tt, Ny, Fx) in hist.items():
    minutes = MISSIONS[name][2]; scale = minutes * 60 / SIM_S
    blocks = []
    for pattern, series in (("brace_y", Ny), ("brace_x", Fx)):
        merged = {}
        for rng_, mean, count in chronos.rainflow(series[::2]):
            if rng_ < 0.2:
                continue
            key = (round(mean, 1), round(rng_ / 2, 1))
            merged[key] = merged.get(key, 0.0) + count * scale
        blocks += [chronos.Block(pattern, m_, a_, c_, f"{name}: {pattern} rainflow") for (m_, a_), c_ in merged.items()]
    spectra[name] = chronos.LoadSpectrum(name, minutes * 60, blocks, ["brace_y", "brace_x"], notes=f"gait model, {SIM_S:.0f} s simulated, scaled to {minutes:.0f} min")
    spectra[name].save(RUNS / f"spectrum_{name.split(' ')[0]}_{name.split(' ')[1]}.json")
    fig = spectra[name].plot()
unit_cases = {}
for pattern, f in {"brace_y": dict(fy=-100.0), "brace_x": dict(fx=100.0)}.items():
    um = leg_model([talos.Force("foot", **f)], f"unit_{pattern}"); um.mesh(RUNS / f"unit_{pattern}")
    unit_cases[pattern] = (um.solve(RUNS / f"unit_{pattern}"), 100.0)
    print(pattern, unit_cases[pattern][0].status, round(unit_cases[pattern][0].metrics["max_von_mises"], 2), "MPa per 100 N")
CURVE = talos.FatigueCurve("PA12-CF (assumed)", sigma_f=110.0, b=-0.10, ultimate=90.0, source="assumed; coupon tests needed")
fatigue = {k: talos.assess_fatigue(unit_cases, sp.to_dict(), CURVE, workdir=RUNS / f"fatigue_{k.split(' ')[0]}_{k.split(' ')[1]}") for k, sp in spectra.items()}
life = pd.DataFrame({k: {"mission_min": MISSIONS[k][2], "damage_per_mission": f.result.metrics["damage_per_pass"],
                         "missions_to_failure": f.result.metrics["passes_to_failure"], "hours_to_failure": f.result.metrics["hours_to_failure"]} for k, f in fatigue.items()}).T
life

In [ ]:
worst_m = life["damage_per_mission"].astype(float).idxmax()
tviz.show(tviz.plot_damage(fatigue[worst_m], unit_cases["brace_y"][0].artifacts["mesh"]))
damage = {k: f.result.metrics["damage_per_pass"] for k, f in fatigue.items()}
hours = {k: MISSIONS[k][2] / 60 for k in MISSIONS}
usage = {"cold 150 mm flue, 15 m": 0.5, "tight 120 mm flue with elbows": 0.3, "clean 180 mm duct, horizontal": 0.2}
rate = sum(usage[k] * damage[k] for k in usage) / sum(usage[k] * hours[k] for k in usage)
sim = chronos.simulate_life(damage, hours, usage, n_flights=3000, seed=0)
fig = sim.plot()
print(f"usage {usage}: damage per 1000 h {rate * 1000:.3g} -> " + (f"{1 / rate:.0f} h to failure" if rate > 1e-9 else "not life-limiting (> 1e9 h)"))

## 8. Version 2 — transport into the chimney

Persephone travels coiled: every joint at 360°/12 = 30° of yaw makes a ring around the tether drum, the
legs folded, in a round carry case with a handle — one person carries it. At the hearth the **insertion
guide**, a quarter-bend half-pipe, is set into the flue mouth: the crawler walks out of the case along the
guide and turns up into the flue.

The insertion sequence from the sketch, as forces:
1. **push in** (*wciskanie*): the segments already braced inside push the trailing ones, which still drag
   on the hearth floor and over the guide — how many segments must be inside before the chain can pull
   the rest;
2. **anchor and pull** (*kotwiczenie i ciągnięcie*): inside the vertical run the braced segments carry
   the hanging ones plus the tether (the bracing margins of section 2);
3. **emergency retreat** (*wycofanie awaryjne*): the drum winch reels the dead crawler out at
   `F_PULLOUT` — winch torque and retrieval time.

In [ ]:
TUCK = dict(hip_angle_deg=-30.0, knee_angle_deg=120.0)                          # legs folded for transport
kit = myropod.generate(version=2, **TUCK)
case = myropod.generate(part="case", **TUCK)
guide = myropod.generate(part="guide", guide_diameter=D_WORK)
cad["case"] = case.export(RUNS / "cad" / "case", stl_tolerance=0.05); cad["guide"] = guide.export(RUNS / "cad" / "guide", stl_tolerance=0.05)
R_COIL = myropod.coil_radius(p)
TETHER_LEN_M = 30.0
m_case = case.volume * RHO_PA12CF / 1000
m_tether = TETHER_LEN_M * TETHER_G_PER_M / 1000
m_winch = 0.9                                                                     # drum motor + gearbox + slip ring (input)
M_KIT = M_CRAWLER + m_case + m_tether + m_winch
kit_tab = pd.Series({"coil radius (centre line) [mm]": R_COIL, "case outer diameter [mm]": kit.dimensions[0], "case height with handle [mm]": kit.dimensions[2],
                     "crawler [kg]": M_CRAWLER, "case [kg]": m_case, f"tether {TETHER_LEN_M:.0f} m [kg]": m_tether, "winch [kg]": m_winch, "kit [kg]": M_KIT,
                     "guide diameter / bend radius [mm]": f"{D_WORK:.0f} / {p['guide_radius']:.0f}", "coil yaw per joint [deg]": myropod.coil_yaw_deg(p),
                     "coil yaw within the joint range": myropod.coil_yaw_deg(p) <= JOINT_RANGE_DEG})
display(kit_tab.to_frame("value"))
dviz.show(dviz.plot3d(kit))

In [ ]:
dviz.show(dviz.plot3d(guide, crawler))

In [ ]:
MU_FLOOR = 0.45                                                                   # feet dragged over the hearth and the guide (input)
rows_ins = {}
for k in range(1, N + 1):                                                         # k segments inside and braced, N - k trailing
    inside = bracing(M_CRAWLER, D=D_WORK, height_m=0.0)
    push_capacity = MU_SOOT * inside["N_leg_N"] * STANCE_FRACTION * 4 * k           # friction the braced legs can transmit along the flue
    drag = MU_FLOOR * (N - k) * W_SEG                                              # trailing segments dragged on the floor
    lift = k * W_SEG                                                              # the inside part is already hanging on its own friction
    rows_ins[k] = {"braced segments": k, "trailing drag [N]": drag, "push/pull capacity [N]": push_capacity, "needs [N]": drag + lift * 0 + 0, "ok": push_capacity >= drag * SF_SLIP}
ins = pd.DataFrame(rows_ins).T
k_min = int(ins.index[ins["ok"].astype(bool)].min())
DRUM_R = p["drum_diameter"] / 2000
T_WINCH = F_PULLOUT * DRUM_R
V_RETREAT = 0.2
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ins["braced segments"], ins["trailing drag [N]"] * SF_SLIP, label=f"trailing drag x SF {SF_SLIP}"); ax.plot(ins["braced segments"], ins["push/pull capacity [N]"], label="friction the braced segments can transmit")
ax.axvline(k_min, color="#c62828", ls="--", label=f"anchor and pull from {k_min} segments inside"); ax.set(xlabel="segments inside the flue", ylabel="N", title="push in → anchor and pull"); ax.legend(); ax.grid(alpha=0.3)
print(f"push-in: the chain can pull itself once {k_min} segments are braced | emergency retreat: {F_PULLOUT:.0f} N on the tether, winch torque {T_WINCH:.1f} Nm at the {p['drum_diameter']:.0f} mm drum, {HEIGHT_M / V_RETREAT / 60:.1f} min from {HEIGHT_M:.0f} m at {V_RETREAT} m/s, {F_PULLOUT * V_RETREAT:.0f} W")

In [ ]:
# the case: carried by its handle (1.5 g), dropped flat on its feet (a 0.3 m fall onto a hard floor as an equivalent 6 g)
reach_t = myropod.leg_reach(myropod.resolve(**TUCK))
H_CASE = p["seg_height"] + reach_t["drop_z"] + 8
r_in = myropod.case_radius(myropod.resolve(**TUCK))
CASE_REGIONS = [talos.SurfacesOnPlane("floor_top", "z", p["case_wall"]), talos.SurfacesOnPlane("feet", "z", -8.0), talos.SurfacesOnPlane("handle_top", "z", H_CASE + 60.0)]
A_FLOOR = math.pi * (r_in**2 - (p["drum_diameter"] / 2)**2)
def case_model(supports, loads, name):
    return talos.StructuralModel(cad["case"].artifacts["step"], "mm-N-MPa", PA12CF, CASE_REGIONS, supports, loads, talos.MeshSettings(element_size=6.0, min_element_size=2.0), name=name)
case_model([talos.FixedSupport("feet")], [talos.Pressure("floor_top", 0.001)], "mesh").mesh(RUNS / "case_mesh", progress=True)
CASE_CASES = {
    "carried by the handle, 1.5 g": ([talos.FixedSupport("handle_top")], [talos.Pressure("floor_top", 1.5 * (M_KIT - m_case) * G / A_FLOOR)]),
    "dropped flat on its feet, 6 g": ([talos.FixedSupport("feet")], [talos.Pressure("floor_top", 6.0 * (M_KIT - m_case) * G / A_FLOOR)]),
}
case_results = {}
for name, (sup, loads) in tqdm(CASE_CASES.items()):
    key = "case_" + name.split(",")[0].replace(" ", "_")
    mdl = case_model(sup, loads, key); mdl.mesh(RUNS / key); case_results[name] = mdl.solve(RUNS / key)
case_tab = pd.DataFrame({k: {"max_von_mises_MPa": r.metrics.get("max_von_mises"), "SF_yield": r.metrics.get("safety_factor_yield"), "deflection_mm": r.metrics.get("max_displacement")} for k, r in case_results.items()}).T
display(case_tab.round(2))
tviz.show(tviz.plot_results(case_results["carried by the handle, 1.5 g"], field="von_mises"))

## 9. Version 3 — two pincers on the head

Two jaws on the head's cheeks, each on its own servo, closing in front of the camera: for pulling a bird's
nest or a soot clump out of the way, or carrying a sensor up the flue. The grip force follows from the jaw
servo torque and the jaw length; the price is the head's moment on the first joint (the head, the jaws, an
object held ahead), which the first pitch servo must hold while the chain is braced.

In [ ]:
v3 = myropod.generate(version=3)
pincer = myropod.generate(part="pincer")
cad["pincer"] = pincer.export(RUNS / "cad" / "pincer", stl_tolerance=0.02)
JAW_SERVO_NM = SERVOS.loc["mini 32 g", "stall_Nm"]
L_JAW = p["pincer_length"] / 1000
F_TIP = JAW_SERVO_NM / L_JAW                                                      # one jaw's tip force at stall
F_GRIP = F_TIP                                                                    # two opposing jaws: the object sees F_TIP from each side
M_OBJECT = 0.3                                                                    # a nest or a clump held in the jaws [kg] (input)
NEEDED_PULL = 20.0                                                                # pulling a nest apart (input)
m_head = head.volume * RHO_PA12CF / 1000 + 0.069
m_jaws = 2 * (pincer.volume * RHO_PA12CF / 1000 + SERVOS.loc["mini 32 g", "mass_g"] / 1000)
x_head = p["head_length"] / 2 + p["joint_gap"] / 2                                # head centre ahead of the first joint [mm]
x_object = x_head + p["head_length"] / 2 + p["pincer_length"] * 0.8
M_JOINT = (m_head * x_head + m_jaws * (x_head + p["head_length"] / 2) + M_OBJECT * x_object) / 1000 * G
v3_tab = pd.Series({"jaw servo [Nm]": JAW_SERVO_NM, "jaw length [mm]": p["pincer_length"], "tip force per jaw [N]": F_TIP, "grip force [N]": F_GRIP, "needed pull [N]": NEEDED_PULL, "grip SF": F_GRIP / NEEDED_PULL,
                    "head + jaws [kg]": m_head + m_jaws, "crawler v3 [kg]": M_CRAWLER + m_jaws, f"moment on joint 1 with a {M_OBJECT} kg object [Nm]": M_JOINT,
                    "first pitch servo [Nm]": JOINT_SERVO["stall_Nm"], "joint 1 SF": JOINT_SERVO["stall_Nm"] / M_JOINT, "length v3 [mm]": v3.dimensions[0]})
display(v3_tab.to_frame("value").round(3))
dviz.show(dviz.plot3d(v3))

In [ ]:
Lp, th = p["pincer_length"], p["pincer_thickness"]
PINCER_REGIONS = [talos.SurfacesInBox("pivot", (-3.0, -3.0, -th, 3.0, 3.0, th)), talos.SurfacesInBox("tip", (Lp - 8, -14, -th, Lp + 1, 2, th))]
def pincer_model(loads, name):
    return talos.StructuralModel(cad["pincer"].artifacts["step"], "mm-N-MPa", PA12CF, PINCER_REGIONS, [talos.FixedSupport("pivot")], loads, talos.MeshSettings(element_size=1.2, min_element_size=0.5), name=name)
PINCER_CASES = {"grip at servo stall": dict(fy=F_TIP), "grip + the object twisted (half the grip sideways)": dict(fy=F_TIP, fz=0.5 * F_TIP), "pulling a nest (tip along the jaw)": dict(fy=0.5 * F_TIP, fx=-NEEDED_PULL)}
pincer_model([talos.Force("tip", fy=1.0)], "mesh").mesh(RUNS / "pincer_mesh", progress=True)
pincer_results = {}
for name, f in tqdm(PINCER_CASES.items()):
    key = "pincer_" + name.split(" ")[0]
    mdl = pincer_model([talos.Force("tip", **f)], key); mdl.mesh(RUNS / key); pincer_results[name] = mdl.solve(RUNS / key)
pincer_tab = pd.DataFrame({k: {"load": ", ".join(f"{kk} {vv:.0f} N" for kk, vv in PINCER_CASES[k].items()), "max_von_mises_MPa": r.metrics.get("max_von_mises"), "SF_yield": r.metrics.get("safety_factor_yield"), "deflection_mm": r.metrics.get("max_displacement")} for k, r in pincer_results.items()}).T
display(pincer_tab.round(2))
tviz.show(tviz.plot_results(pincer_results["grip + the object twisted (half the grip sideways)"], field="von_mises"))

In [ ]:
compare = pd.DataFrame({
    "v1 crawler": {"mass [kg]": M_CRAWLER, "length [mm]": LENGTH, "bracing flues [mm]": f"{D_MIN_BRACE:.0f}–{D_MAX_BRACE:.0f}", "worst leg servo SF (150 mm, 15 m)": brace_tab.loc["150 mm flue, 15 m up", "servo SF"],
                   "climb power at 15 m [W]": power(HEIGHT_M)["total_W"], "minutes in a 120 °C flue": thermal_tab["wall 120 °C"]["minutes to the limit"], "pull-out force [N]": F_PULLOUT, "adds": "—"},
    "v2 transport kit": {"mass [kg]": M_KIT, "length [mm]": kit.dimensions[0], "bracing flues [mm]": "same crawler", "worst leg servo SF (150 mm, 15 m)": brace_tab.loc["150 mm flue, 15 m up", "servo SF"],
                         "climb power at 15 m [W]": power(HEIGHT_M)["total_W"], "minutes in a 120 °C flue": thermal_tab["wall 120 °C"]["minutes to the limit"], "pull-out force [N]": F_PULLOUT,
                         "adds": f"case Ø{kit.dimensions[0]:.0f} mm, {TETHER_LEN_M:.0f} m tether on the drum, winch {T_WINCH:.1f} Nm, guide for {D_WORK:.0f} mm, pull from {k_min} segments in"},
    "v3 with pincers": {"mass [kg]": M_CRAWLER + m_jaws, "length [mm]": v3.dimensions[0], "bracing flues [mm]": f"{D_MIN_BRACE:.0f}–{D_MAX_BRACE:.0f}",
                        "worst leg servo SF (150 mm, 15 m)": SERVOS.loc[choice, "stall_Nm"] / max(bracing(M_CRAWLER + m_jaws)["tau_hip_Nm"], bracing(M_CRAWLER + m_jaws)["tau_knee_Nm"]),
                        "climb power at 15 m [W]": power(HEIGHT_M)["total_W"] * (M_CRAWLER + m_jaws) / M_CRAWLER, "minutes in a 120 °C flue": thermal_tab["wall 120 °C"]["minutes to the limit"], "pull-out force [N]": F_PULLOUT + m_jaws * G,
                        "adds": f"grip {F_GRIP:.0f} N, joint-1 SF {JOINT_SERVO['stall_Nm'] / M_JOINT:.2f} with a {M_OBJECT} kg object"},
}).T
compare

## 10. Print a segment (Mellonia) and export

The segment shell prints open side down… it has none: it prints as a closed hollow box on its belly rib
(rotated nothing), with the hatch on top; the hollow needs the slicer's bridging over 39 mm.

In [ ]:
prn = mellonia.slice_stl(cad["segment"].artifacts["stl"], GENERIC_PLA_0_2MM, mellonia.Orientation(), RUNS / "print_segment")
print(prn)
if prn.ok:
    mviz.show(mviz.plot_toolpath(prn))

In [ ]:
summary_doc = {"family": "Myropods", "member": "Persephone", "design": {"file": "designs/myropod.py", "parameters": p},
               "mass_kg": {"crawler": M_CRAWLER, "kit": M_KIT, "v3": M_CRAWLER + m_jaws}, "per_segment_g": per_segment.to_dict(), "servo": {"leg": choice, **SERVOS.loc[choice].to_dict(), "joint": JOINT_SERVO},
               "fit": {"bracing_range_mm": [D_MIN_BRACE, D_MAX_BRACE], "min_bend_radius_mm": R_MIN_BEND, "mitre_min_mm": D_MIN_MITRE, "flues": flues},
               "bracing": brace_tab.to_dict(orient="index"), "climb": {"speed_m_s": v_climb, "minutes_15m": climb_time_min, "power": rows},
               "static": static.to_dict(orient="index"), "thermal": thermal_tab, "links": budget, "leg_modes_hz": freqs,
               "life": life.to_dict(orient="index"), "damage_per_1000h": rate * 1000,
               "v2": {"kit": {k: (float(v) if isinstance(v, (int, float, np.floating)) else str(v)) for k, v in kit_tab.items()}, "insertion_min_segments": k_min, "winch_Nm": T_WINCH, "case": case_tab.to_dict(orient="index")},
               "v3": {k: (float(v) if isinstance(v, (int, float, np.floating)) else str(v)) for k, v in v3_tab.items()}, "pincer": pincer_tab.to_dict(orient="index"),
               "compare": compare.to_dict(orient="index")}
(RUNS / "persephone.json").write_text(json.dumps(summary_doc, indent=2, default=str))
print("written:", sorted(x.name for x in RUNS.iterdir())[:16], "...")

**Next steps an engineer would take:** measure friction of the real pads on real soot (it sets every
bracing number); replace the servo table with torque–speed curves at the voltage the segments actually
see; test one segment with its four legs in a 150 mm pipe (does it hold at the computed `N`?); and, if
the hanging-chain case or the slip-catch case is the limiting one, thicken the joint necks or the tibia —
a revision in a `vegeta.core` workspace, or a campaign with the leg's `damage_per_1000h` as the objective.
Cleopatra, the three-segment walker of the same family, is notebook 18.